In [1]:

import time, itertools, random
import pandas as pd
import numpy as np

def load_ctx_drug(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, index_col=0)
    df.columns = df.columns.str.strip()
    df.index = df.index.astype(str).str.strip()
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

def pairwise_tissue_sets(df: pd.DataFrame):
    """Return drugs, tissues, and S[a][b] = set of tissues where a>b."""
    drugs = list(df.columns)
    tissues = list(df.index)
    n = len(drugs)
    S = [[set() for _ in range(n)] for __ in range(n)]
    for i, a in enumerate(drugs):
        for j, b in enumerate(drugs):
            if i==j: continue
            wins = set()
            for t in tissues:
                va, vb = df.at[t, a], df.at[t, b]
                if pd.isna(va) or pd.isna(vb):
                    continue
                if va > vb:
                    wins.add(t)
            S[i][j] = wins
    return drugs, tissues, S

def path_intersection_prefix(order_idxs, S, lam):
    """Given index order and S[i][j], return the longest λ-consistent prefix path and its common tissues S*."""
    if not order_idxs: return [], set(), []
    S_common = None
    adj_sets = []
    path = [order_idxs[0]]
    for i,j in zip(order_idxs[:-1], order_idxs[1:]):
        Sab = S[i][j]
        S_common = Sab if S_common is None else (S_common & Sab)
        if len(S_common) >= lam:
            path.append(j)
            adj_sets.append(Sab & S_common)
        else:
            break
    return path, (S_common if S_common is not None else set()), adj_sets

def print_result(tag, elapsed_s, order, S):
    print(f"[{tag}] time = {elapsed_s:.3f} s | path length = {len(order)} | |S| = {len(S)}")
    if order:
        print("  Path:", " > ".join(order))
    else:
        print("  Path: (empty)")


In [2]:
# ===== TA-Kemeny (exact, small n) =====
import itertools
import pandas as pd

def ta_kemeny_path(df: pd.DataFrame, lam: int, max_n: int = 10):
    """
    Exact (factorial) search over permutations, but instead of requiring a full
    n-drug chain, we accept the LONGEST λ-consistent PREFIX of each permutation.
    We then pick the best by (prefix length, score). This avoids 'empty' results
    when no full-length λ-consistent order exists.
    """
    drugs, tissues, S = pairwise_tissue_sets(df)
    n = len(drugs)
    if n > max_n:
        raise RuntimeError(f"TA-Kemeny exact is capped at n<={max_n}; current n={n}. Subset columns before calling.")

    best_order_idx, best_len, best_score, best_S = None, 0, -1, set()

    for perm in itertools.permutations(range(n)):
        S_common = None
        prefix = [perm[0]]  # a single drug is trivially feasible
        score = 0
        ok = True

        # grow the prefix while λ holds
        for i, j in zip(perm[:-1], perm[1:]):
            Sab = S[i][j]                 # tissues where i > j
            new_S = Sab if S_common is None else (S_common & Sab)

            if len(new_S) < lam:
                ok = False
                break

            # accept this step
            S_common = new_S
            prefix.append(j)
            # scoring: reward edges supported under the current global intersection
            score += len(Sab & S_common)

        # Evaluate this permutation's best feasible prefix
        plen = len(prefix)
        if plen > best_len or (plen == best_len and score > best_score):
            best_order_idx = prefix[:]
            best_len = plen
            best_score = score
            best_S = set(S_common) if S_common is not None else set()

        # small pruning: if even a perfect continuation can't beat best_len, skip
        # (omitted here for simplicity; n<=10 makes it unnecessary)

    if best_order_idx is None or best_len == 0:
        return [], set(), []
    return [drugs[i] for i in best_order_idx], best_S, []

def run_takem(csv_path, lam=30, max_n=10):
    df = load_ctx_drug(csv_path)
    if df.shape[1] > max_n:
        df = df.iloc[:, :max_n]
        print(f"[TA-Kemeny] Warning: using only first {max_n} drugs (n={df.shape[1]}).")
    t0 = time.time()
    order, S_common, _ = ta_kemeny_path(df, lam, max_n=max_n)
    elapsed = time.time() - t0
    print_result("TA-Kemeny (exact, small n, robust prefix)", elapsed, order, S_common)


In [3]:
run_takem("efficacy-diabetes.csv", lam=30, max_n=12)

[TA-Kemeny] Warning: using only first 12 drugs (n=12).
[TA-Kemeny (exact, small n, robust prefix)] time = 864.807 s | path length = 8 | |S| = 30
  Path: CLOZAPINE > ASPIRIN > CYCLOSPORINE > METHOTREXATE > SORAFENIB > CENISERTIB > FLUOROURACIL > PACLITAXEL


In [4]:
run_takem("efficacy-bpco.csv", lam=30, max_n=12)

[TA-Kemeny] Warning: using only first 12 drugs (n=12).
[TA-Kemeny (exact, small n, robust prefix)] time = 778.981 s | path length = 6 | |S| = 38
  Path: ASPIRIN > DOXORUBICIN HYDROCHLORIDE > CISPLATIN > RESVERATROL > FLUOROURACIL > RITUXIMAB


In [ ]:
run_takem("efficacy-bpco.csv", lam=30, max_n=13)